In [ ]:
import os
import pandas as pd
import numpy as np

"""
Resample ICU parquet datasets to an **hourly** grid, but only for timestamps
inside ± 6 h windows around each patient‑specific hemoglobin (Hb) measurement.

**2025‑06‑24 update — per‑encounter windows**
• Hb windows are now computed **separately for every encounterId**.
• `fast_resample_and_reindex()` accepts a pre‑built *MultiIndex* of the
  (encounterId, time) pairs that are allowed, and reindexes strictly to it.
• Each source DataFrame is inner‑joined with this valid‑pair table *before*
  heavy type conversion, further trimming RAM.
"""

# ────────────────────────────── configuration ──────────────────────────────
SAMPLING_INTERVAL = "1h"
WINDOW_HOURS = 6

DATA_PATH = "/home/lkapral/hb/data/"
OUT_PATH = "/home/lkapral/hb/data/resampled/"

LAB_RESULTS_FILE = "lab-results-1min-pivot-20241008.parquet"
HB_COLUMN = "hemoglobin_g/dl"

# Columns to retain from each parquet -------------------------------------------------------------
COLMAP = {
    "blood-pressure-1min-pivot-20240807.parquet": [
        "encounterId", "utcChartTime",
        "blood_pressure_systolic_mmHg",
        "blood_pressure_mean_mmHg",
        "blood_pressure_diastolic_mmHg",
    ],
    "catecholamines-1min-pivot-20240719.parquet": [
        "encounterId", "utcChartTime",
        "norepinephrine_µg/kg/min",
        "dobutamine_µg/kg/min",
        "vasopressin_IE/h",
    ],
    "intake-1min-pivot-20240719.parquet": [
        "encounterId", "utcChartTime",
        "fluids_ml", "colloids_ml",
    ],
    "lab-results-1min-pivot-20241008.parquet": [
        "encounterId", "utcChartTime",
        "hemoglobin_g/dl", "lactate_mmol/l",
        "fibrinogen_mg/dl", "platelet_count_G/l",
        "base_excess_mmol/l",
    ],
    "sitecare-1min-pivot-20240719.parquet": [
        "encounterId", "utcChartTime", 'harnk_ml',
        "jackson-pratt_ml", "redon-drain_ml",
        "robinson-drain_ml", "thorax-drain_ml", "easy-flow_ml",
    ],
    "vitals_wide.parquet": [
        "encounterId", "utcChartTime",
        "heart_rate", "respiratory_rate",
    ],
    "blood_wide.parquet": [
        "encounterId", "utcChartTime",
        "blood_input", 
    ],
        "spo2.parquet": [
        "encounterId", "utcChartTime",
        "spo2",
    ],
}
CONST_COL_CANDIDATES = {"age", "sex_or_gender", "sex_or_gender_numeric"}

def const_cols_for(fname: str) -> list[str]:
    return [c for c in COLMAP[fname] if c in CONST_COL_CANDIDATES]

# ───────────────────────── helper utilities ────────────────────────────────

def downcast_df(df: pd.DataFrame) -> pd.DataFrame:
    df2 = df.copy()
    for col in df2.columns:
        if pd.api.types.is_float_dtype(df2[col]):
            df2[col] = pd.to_numeric(df2[col], downcast="float")
        elif pd.api.types.is_integer_dtype(df2[col]):
            df2[col] = pd.to_numeric(df2[col], downcast="integer")
        elif pd.api.types.is_object_dtype(df2[col]):
            nunq = df2[col].nunique(dropna=False)
            if 0 < nunq < len(df2[col]) / 2:
                df2[col] = df2[col].astype("category")
    return df2


def fast_resample_and_reindex(
    df: pd.DataFrame,
    *,
    time_column: str,
    interval: str,
    allowed_pairs_index: pd.MultiIndex,
    const_cols: list[str] | None = None,
) -> pd.DataFrame:
    """Resample `df` on `interval`, confined to `allowed_pairs_index`.

    `allowed_pairs_index` MUST be a MultiIndex with levels
    (encounterId, time_column). Only those pairs will appear in the output.
    """
    if "encounterId" not in df.columns:
        raise ValueError("'encounterId' column is required.")

    df2 = df.copy()
    df2[time_column] = pd.to_datetime(df2[time_column]).dt.floor(interval)

    # Early inner‑join with allowed pairs for speed
    # Inner-join with the allowed (encounterId, time) pairs — give the helper Series a
    # dummy name to avoid the ValueError about unnamed Series.
    keep_idx = pd.Series(1, index=allowed_pairs_index, name="__keep")
    df2 = (
        df2.set_index(["encounterId", time_column])
        .join(keep_idx, how="inner")
        .drop(columns="__keep")
        .reset_index()
    )
    if df2.empty:
        return pd.DataFrame(columns=df.columns)

    const_cols = [c for c in (const_cols or []) if c in df2.columns]
    const_df = (
        df2[["encounterId"] + const_cols]
        .drop_duplicates("encounterId")
        .set_index("encounterId")
        if const_cols
        else None
    )

    agg_spec = {
        "blood_pressure_diastolic_mmHg": "mean",
        "blood_pressure_mean_mmHg": "mean",
        "blood_pressure_systolic_mmHg": "mean",
        "dobutamine_µg/kg/min": "mean",
        "fibrinogen_mg/dl": "mean",
        "lactate_mmol/l": "mean",
        "norepinephrine_µg/kg/min": "mean",
        "platelet_count_G/l": "mean",
        "vasopressin_IE/h": "mean",
        "colloids_ml": "sum",
        "easy-flow_ml": "sum",
        "fluids_ml": "sum",
        "jackson-pratt_ml": "sum",
        "redon-drain_ml": "sum",
        "robinson-drain_ml": "sum",
        "thorax-drain_ml": "sum",
        "harnk_ml": "sum",
        "hemoglobin_g/dl": "min",
        "base_excess_mmol/l": "min",
        "heart_rate": "mean",
        "respiratory_rate": "mean",
        "spo2": "mean",
        "blood_input": "sum",
        
    }
    valid_agg = {c: op for c, op in agg_spec.items() if c in df2.columns}

    grouped = (
        df2
        .groupby([
            "encounterId", pd.Grouper(key=time_column, freq=interval)
        ], observed=True)
        .agg(valid_agg)
    )
    grouped.index.rename(["encounterId", time_column], inplace=True)

    # Reindex to *exactly* allowed pairs
    grouped = grouped.reindex(allowed_pairs_index)

    sums = [c for c, op in valid_agg.items() if op == "sum"]
    overwise = [c for c in grouped.columns if c not in sums]

    result = grouped.reset_index()
    if const_df is not None:
        result = result.merge(const_df.reset_index(), on="encounterId", how="left")
    return result

# ───────────────────────── build per‑encounter windows ─────────────────────
print("[0/..?] Building per‑encounter Hb windows …")
lab = pd.read_parquet(os.path.join(DATA_PATH, LAB_RESULTS_FILE), columns=["encounterId", "utcChartTime", HB_COLUMN])
lab["utcChartTime"] = pd.to_datetime(lab["utcChartTime"]).dt.floor(SAMPLING_INTERVAL)
lab = lab[lab[HB_COLUMN].notna()]

TDELTA = pd.Timedelta(hours=WINDOW_HOURS)
allowed_pairs: list[tuple[int, pd.Timestamp]] = []

for enc_id, times in lab.groupby("encounterId"):
    unique_ts = times["utcChartTime"].unique()
    # Build & merge windows per encounter
    raw = sorted([(t - TDELTA, t) for t in unique_ts], key=lambda p: p[0])
    merged: list[list[pd.Timestamp]] = []
    for start, end in raw:
        if not merged or start > merged[-1][1]:
            merged.append([start, end])
        else:
            merged[-1][1] = max(merged[-1][1], end)
    for s, e in merged:
        allowed_pairs.extend([(enc_id, ts) for ts in pd.date_range(s, e, freq=SAMPLING_INTERVAL)])

allowed_pairs_index = pd.MultiIndex.from_tuples(allowed_pairs, names=["encounterId", "utcChartTime"])
print(f"   → Generated {len(allowed_pairs_index)} (encounter,time) pairs across {lab['encounterId'].nunique()} encounters.")

# ───────────────────────── main loop over files ────────────────────────────
os.makedirs(OUT_PATH, exist_ok=True)
for i, fname in enumerate(COLMAP, start=1):
    cols = COLMAP[fname]
    const_cols = const_cols_for(fname)

    print(f"[{i}/{len(COLMAP)}] Processing {fname} …")
    path = os.path.join(DATA_PATH, fname)
    if not os.path.exists(path):
        print("   → File not found – skipping.")
        continue

    df = pd.read_parquet(path, columns=cols)
    df["utcChartTime"] = pd.to_datetime(df["utcChartTime"]).dt.floor(SAMPLING_INTERVAL)

    # Inner join with allowed pairs early
    keep_idx = pd.Series(1, index=allowed_pairs_index, name="__keep")
    df = (
        df.set_index(["encounterId", "utcChartTime"])\
          .join(keep_idx, how="inner")\
          .drop(columns="__keep")\
          .reset_index()
    )
    if df.empty:
        print("   → No matching rows – skipping.")
        continue

    df["encounterId"] = pd.to_numeric(df["encounterId"], errors="coerce")
    df.dropna(subset=["encounterId"], inplace=True)
    df["encounterId"] = df["encounterId"].astype(int)

    df = downcast_df(df)

    df_res = fast_resample_and_reindex(
        df,
        time_column="utcChartTime",
        interval=SAMPLING_INTERVAL,
        allowed_pairs_index=allowed_pairs_index,
        const_cols=const_cols,
    )

    out_file = os.path.join(OUT_PATH, fname.replace(".parquet", "_resampled.parquet"))
    print(f"   → Writing {out_file}  (shape = {df_res.shape})")
    df_res.to_parquet(out_file)

print("All datasets processed! 🚀")


In [ ]:
!python --version

In [ ]:
import pandas as pd
CIS = pd.read_parquet('data/CIS_D_Encounter_20250321T1443-all.parquet')

In [ ]:
CIS = CIS.rename(columns={"gender": "sex_or_gender"})
CIS = CIS.reset_index()
CIS.to_parquet("data/CIS.parquet", index=False)

In [ ]:
CIS

In [ ]:

# Load & Process Intervention Data

inter_path = 'data/intervention-type-20251201.parquet'

inter = pd.read_parquet(inter_path)


In [ ]:
inter['encounterId'].nunique()

In [ ]:
inter['ptDemographicId'].nunique()

In [ ]:
inter

In [ ]:
inter['encounterId'].nunique()

In [ ]:
import pandas as pd
demo = pd.read_parquet('data/demographic-1min-pivot-20240719.parquet')

In [ ]:
import pandas as pd
labs = pd.read_parquet('data/lab-results-1min-pivot-20241008.parquet')

In [ ]:
labs['encounterId'].nunique()

In [ ]:
labs

In [ ]:
labs.loc[labs['hemoglobin_g/dl']>0,'encounterId'].nunique()

In [ ]:
labs[labs[h]]

In [ ]:
inter['encounterId'].nunique()/labs.loc[labs['hemoglobin_g/dl']>0,'encounterId'].nunique()

In [ ]:
inter.loc[inter['longLabel']!='Dringlichkeit.Dringlichkeit','longLabel'].value_counts()

In [ ]:
inter.loc[inter['longLabel']=='Dringlichkeit.Dringlichkeit','verboseFormEscaped'].value_counts()

In [ ]:
COLMAP = {
    "blood-pressure-1min-pivot-20240807.parquet": [
        "encounterId", "utcChartTime",
        "blood_pressure_systolic_mmHg",
        "blood_pressure_mean_mmHg",
        "blood_pressure_diastolic_mmHg",
    ],
    "catecholamines-1min-pivot-20240719.parquet": [
        "encounterId", "utcChartTime",
        "norepinephrine_µg/kg/min",
        "dobutamine_µg/kg/min",
        "vasopressin_IE/h",
    ],
    "intake-1min-pivot-20240719.parquet": [
        "encounterId", "utcChartTime",
        "fluids_ml", "colloids_ml",
    ],
    "lab-results-1min-pivot-20241008.parquet": [
        "encounterId", "utcChartTime",
        "hemoglobin_g/dl", "lactate_mmol/l",
        "fibrinogen_mg/dl", "platelet_count_G/l",
        "base_excess_mmol/l",
    ],
    "sitecare-1min-pivot-20240719.parquet": [
        "encounterId", "utcChartTime", 'harnk_ml',
        "jackson-pratt_ml", "redon-drain_ml",
        "robinson-drain_ml", "thorax-drain_ml", "easy-flow_ml",
    ],
    "vitals_wide.parquet": [
        "encounterId", "utcChartTime",
        "heart_rate", "respiratory_rate",
    ],
    "blood_wide.parquet": [
        "encounterId", "utcChartTime",
        "blood_input", 
    ],
    "spo2.parquet": [
        "encounterId", "utcChartTime",
        "spo2", 
    ],
}

In [ ]:
import os
import pandas as pd
import numpy as np

test = pd.read_parquet('data/spo2-high-density-20251103.parquet')

In [ ]:
test.rename

In [ ]:
test = test.rename(columns={"valueNumber_percent": "spo2", "utcmeasurementTime": "utcChartTime",})

In [ ]:
test["utcChartTime"] = (
    pd.to_datetime(test["utcChartTime"])           # Convert to datetime
    .dt.tz_convert("UTC")                        # Ensure UTC timezone
    .dt.strftime("%Y-%m-%d %H:%M:%S%z")          # Format with UTC offset
)

In [ ]:
test.to_parquet('data/spo2.parquet')

In [ ]:
test

In [ ]:
df = pd.read_parquet('data/vitals_wide.parquet')

In [ ]:
df